In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns

def get_csv_header_pandas(filepath):
    # Pandas automatically treats the first row as the header by default
    df = pd.read_csv(filepath) 
    # The .columns attribute holds the header names
    header_list = df.columns.tolist()
    return header_list

def plot_confusion_matrix(cm):
    N = np.sum(cm)
    cm_ratio = cm / N * 100

    annot_labels = np.empty(cm.shape, dtype=object)
    for i in range(cm.shape[0]):
      for i_idx, j in enumerate(range(cm.shape[1])):
        val = cm[i, j]
        pct = cm_ratio[i, j]
        annot_labels[i, j] = f'{val}\n({pct:.1f}%)'

    plt.figure(figsize=(6, 5))
    sns.heatmap(
        cm,
        annot=annot_labels,
        fmt='',
        cmap='Blues',
        cbar=False,
        xticklabels=['Has substructure', 'No Substructure'],
        yticklabels=['Has substructure', 'No Substructure'],
    )

    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title(f'Confusion Matrix ({N} lenses)')
    plt.show()

## Load the answer key

In [ ]:
base_directory = os.getcwd()
answer_key_filename = r"unlabeled_answer_key_v_3_0.csv"
answer_file = os.path.join(base_directory, answer_key_filename)

In [ ]:
print(get_csv_header_pandas(answer_file))

In [ ]:
# Read a specific column by name
actual = pd.read_csv(answer_file, usecols=['substructure_flag']).values.flatten()
actual = np.array(actual, dtype=bool)
uid = pd.read_csv(answer_file, usecols=['uid']).values.flatten()
assert len(actual) == len(uid)

N = len(uid)
print(N)

## Load the submission

In [ ]:
team_name = ""
submission_number = 

In [ ]:
submission_location = os.path.join(base_directory, "Submissions")
submission_filename = f"{team_name}_rung1_submission{str(submission_number)}.csv"
submission_file = os.path.join(submission_location, submission_filename)

In [ ]:
print(get_csv_header_pandas(submission_file))

In [ ]:
# Read a specific column by name
predicted = pd.read_csv(submission_file, usecols=['has_substructure']).values.flatten()
predicted = np.array(predicted, dtype=bool)
ID = pd.read_csv(submission_file, usecols=['ID']).values.flatten()
assert len(predicted) == len(ID)

for i in range(len(ID)):
    try:
        ID[i] = int(ID[i])
    except:
        # if the string start with "strong_lens_"
        ID[i] = int(ID[i][12:])

np.testing.assert_equal(ID, uid)

## Compute Accuracy, Confusion Matrix, Precision

$$\text{Accuracy}=\frac{\text{number of correct predictions}}{\text{total number of lenses}}$$

In [ ]:
correct_predictions = (predicted == actual)
accuracy = np.sum(correct_predictions) / N
print("Accuracy:", accuracy)

$$\text{Confusion matrix}=\left(
\begin{array}{cc}
   TP & FN \\
   FP & TN \\
\end{array} 
\right)
$$
where TP is True Positive, FN is False Negative, etc

In [ ]:
TP = np.sum((predicted == True) & (actual == True))
FN = np.sum((predicted == False) & (actual == True))
FP = np.sum((predicted == True) & (actual == False))
TN = np.sum((predicted == False) & (actual == False))
assert TP + TN + FP + FN == N

cm = np.array([[TP, FN], [FP, TN]])
plot_confusion_matrix(cm)

$$
\text{Precision}=\frac{\text{TP}}{\text{TP}+\text{FP}}
$$

In [ ]:
precision = TP / (TP + FP)
print("Precision:", precision)

# Save the results to file
Results for individual lenses are not accessible by the participants.

In [ ]:
grade_location = os.path.join(base_directory, "Grades")
grade_filename = f"{team_name}_rung1_grades{str(submission_number)}.csv"
grade_file = os.path.join(grade_location, grade_filename)

In [ ]:
results = {
    "ID": ID,
    "has_substructure": actual,
    "predicted": predicted,
    "correct": correct_predictions,
    "": [None] * N,
    "accuracy": [accuracy] + [None] * (N - 1),
    "precision": [precision] + [None] * (N - 1),
    "TP": [TP] + [None] * (N - 1),
    "FN": [FN] + [None] * (N - 1),
    "FP": [FP] + [None] * (N - 1),
    "TN": [TN] + [None] * (N - 1)
}
df = pd.DataFrame(results)

df.to_csv(grade_file, index=False)